# LeetCode 1260 — Shift 2D Grid bằng Python

**Link đề:** https://leetcode.com/problems/shift-2d-grid/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix, Simulation

```python
class Solution:
    def shiftGrid(self, grid: list[list[int]], k: int) -> list[list[int]]:
        ...
```

## Mục tiêu học
1. Hiểu phép "dịch lưới" chính là **xoay vòng (rotate) một dãy** khi ta đọc lưới theo từng hàng.
2. Nắm các công cụ Python: duỗi ma trận (flatten), **slicing âm** `a[-k:]`, `a[:-k]`, phép `%` để quay vòng, `divmod` để đổi chỉ số 1 chiều ↔ 2 chiều.
3. Đi từ **mô phỏng k lần** (dễ nghĩ, chậm) → **duỗi + xoay** → **tính thẳng vị trí mới** bằng công thức chỉ số.
4. Biết thêm kỹ thuật **đảo 3 lần** (three reversals) và `collections.deque.rotate`.
5. Trace, chứng minh, test đối chiếu, lỗi hay gặp (quên `k % total`!), biến thể, bài tập có đáp án.

## 1. Đề bài và ví dụ

Cho lưới `grid` kích thước `m x n` và số nguyên `k`. Thực hiện phép **dịch** `k` lần. Một lần dịch:

- Phần tử ở `grid[i][j]` chuyển sang `grid[i][j + 1]`.
- Phần tử ở cuối hàng `grid[i][n - 1]` chuyển xuống **đầu hàng kế tiếp** `grid[i + 1][0]`.
- Phần tử ở góc cuối `grid[m - 1][n - 1]` quay về **góc đầu** `grid[0][0]`.

Trả về lưới sau `k` lần dịch.

| `grid` | `k` | Output |
|---|---:|---|
| `[[1,2,3],[4,5,6],[7,8,9]]` | 1 | `[[9,1,2],[3,4,5],[6,7,8]]` |
| `[[3,8,1,9],[19,7,2,5],[4,6,11,10],[12,0,21,13]]` | 4 | `[[12,0,21,13],[3,8,1,9],[19,7,2,5],[4,6,11,10]]` |
| `[[1,2,3],[4,5,6],[7,8,9]]` | 9 | `[[1,2,3],[4,5,6],[7,8,9]]` |

**Constraints:** `1 <= m, n <= 50`, `-1000 <= grid[i][j] <= 1000`, `0 <= k <= 100`.

### Nhìn lưới như một dãy dài

Đọc ví dụ 1 theo từng hàng, trái → phải, trên → dưới:

```text
trước: 1 2 3 4 5 6 7 8 9
sau 1: 9 1 2 3 4 5 6 7 8
```

Ba quy tắc của đề gộp lại thành **một** quy tắc: mỗi phần tử tiến 1 bước trong dãy dài, phần tử cuối quay về đầu. Đó là **xoay phải 1 bước**.

Ví dụ 3: dãy dài có 9 phần tử, xoay 9 bước thì mọi phần tử về đúng chỗ cũ → lưới không đổi.

In [1]:
grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat = [x for row in grid for x in row]
print("dãy dài:", flat)
shifted_once = [flat[-1]] + flat[:-1]
print("xoay 1  :", shifted_once)   # [9, 1, 2, 3, 4, 5, 6, 7, 8]

dãy dài: [1, 2, 3, 4, 5, 6, 7, 8, 9]
xoay 1  : [9, 1, 2, 3, 4, 5, 6, 7, 8]


## 2. Nền tảng Python cần biết

### 2.1. Duỗi ma trận (flatten) bằng list comprehension 2 vòng

```python
flat = [x for row in grid for x in row]
```

Đọc từ trái sang phải giống 2 vòng lồng nhau: `for row in grid:` (ngoài) rồi `for x in row:` (trong), lấy `x`.

In [2]:
grid = [[1, 2], [3, 4], [5, 6]]
flat = [x for row in grid for x in row]
print(flat)                 # [1, 2, 3, 4, 5, 6]

# Viết dài tương đương
flat2 = []
for row in grid:
    for x in row:
        flat2.append(x)
print(flat2 == flat)        # True

[1, 2, 3, 4, 5, 6]
True


### 2.2. Slicing với chỉ số âm

- `a[-k:]` = **k phần tử cuối**.
- `a[:-k]` = mọi phần tử **trừ k phần tử cuối**.
- Ghép `a[-k:] + a[:-k]` = đưa k phần tử cuối lên đầu = **xoay phải k bước**.

In [3]:
a = [1, 2, 3, 4, 5]
k = 2
print(a[-k:])               # [4, 5]
print(a[:-k])               # [1, 2, 3]
print(a[-k:] + a[:-k])      # [4, 5, 1, 2, 3]

[4, 5]
[1, 2, 3]
[4, 5, 1, 2, 3]


### 2.3. Phép `%` để "quay vòng"

Xoay một dãy độ dài `total` đúng `total` bước thì về như cũ. Nên xoay `k` bước ≡ xoay `k % total` bước.
`%` cũng dùng để "cuộn" chỉ số vượt quá cuối dãy về đầu: vị trí `p` vượt quá → `p % total`.

In [4]:
total = 9
for k in [0, 1, 9, 10, 23]:
    print(f"k={k:>2} -> k % total = {k % total}")
print((7 + 4) % 9)          # vị trí 7 tiến 4 bước trong dãy 9 phần tử -> 2

k= 0 -> k % total = 0
k= 1 -> k % total = 1
k= 9 -> k % total = 0
k=10 -> k % total = 1
k=23 -> k % total = 5
2


### 2.4. Đổi chỉ số 2 chiều ↔ 1 chiều

Với lưới có `n` cột, ô `(i, j)` nằm ở vị trí `p = i * n + j` trong dãy dài (vì trước nó có `i` hàng đầy, mỗi hàng `n` phần tử, cộng thêm `j`).
Ngược lại: `i = p // n`, `j = p % n`. Hàm `divmod(p, n)` trả về cả hai cùng lúc.

In [5]:
n = 3
for i, j in [(0, 0), (1, 2), (2, 1)]:
    p = i * n + j
    print(f"(i={i}, j={j}) -> p={p} -> divmod -> {divmod(p, n)}")

(i=0, j=0) -> p=0 -> divmod -> (0, 0)
(i=1, j=2) -> p=5 -> divmod -> (1, 2)
(i=2, j=1) -> p=7 -> divmod -> (2, 1)


### 2.5. Cắt dãy dài thành các hàng

`[flat[i:i + n] for i in range(0, len(flat), n)]` — `range(0, len, n)` nhảy từng bước `n` (0, n, 2n, …), mỗi lần cắt một đoạn dài `n`.

In [6]:
flat = [1, 2, 3, 4, 5, 6]
n = 2
print(list(range(0, len(flat), n)))                        # [0, 2, 4]
print([flat[i:i + n] for i in range(0, len(flat), n)])     # [[1, 2], [3, 4], [5, 6]]

[0, 2, 4]
[[1, 2], [3, 4], [5, 6]]


## 3. Cách 1 — Mô phỏng từng lần dịch (đúng theo lời đề)

Mỗi lần dịch: tạo lưới mới, đặt từng phần tử vào vị trí kế tiếp theo 3 quy tắc. Lặp lại `k` lần.

In [7]:
def shift_once(grid):
    m, n = len(grid), len(grid[0])
    new = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if j < n - 1:                 # quy tắc 1: sang phải
                new[i][j + 1] = grid[i][j]
            elif i < m - 1:               # quy tắc 2: cuối hàng -> đầu hàng dưới
                new[i + 1][0] = grid[i][j]
            else:                         # quy tắc 3: góc cuối -> góc đầu
                new[0][0] = grid[i][j]
    return new


def shift_grid_simulate(grid, k):
    for _ in range(k):
        grid = shift_once(grid)
    return grid

print(shift_grid_simulate([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1))
print(shift_grid_simulate([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 9))

[[9, 1, 2], [3, 4, 5], [6, 7, 8]]
[[1, 2, 3], [4, 5, 6], [7, 8, 9]]


**Độ phức tạp Cách 1:** mỗi lần dịch `O(m·n)` → tổng `O(k·m·n)`; bộ nhớ `O(m·n)`.
Với constraints (`k <= 100`, `m·n <= 2500`) vẫn chạy được (~250.000 bước), nhưng nếu `k` lên `10^9` thì không ổn.

## 4. Nút thắt

1. Ta dịch `k` lần dù dịch `total = m·n` lần là **về nguyên trạng** → chỉ cần `k % total` lần.
2. Ta di chuyển **từng bước một**, trong khi có thể nhảy thẳng **k bước** một lần, vì mọi phần tử đều tiến đúng `k` vị trí trong dãy dài.

## 5. Cách 2 — Duỗi, xoay bằng slicing, cắt lại

1. `flat` = dãy dài.
2. `k %= total` (rất quan trọng!).
3. `flat = flat[-k:] + flat[:-k]` nếu `k > 0`.
4. Cắt lại thành `m` hàng, mỗi hàng `n` phần tử.

In [8]:
def shift_grid_flatten(grid, k):
    m, n = len(grid), len(grid[0])
    flat = [x for row in grid for x in row]
    total = m * n
    k %= total
    if k:
        flat = flat[-k:] + flat[:-k]
    return [flat[i:i + n] for i in range(0, total, n)]

print(shift_grid_flatten([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1))
print(shift_grid_flatten([[3, 8, 1, 9], [19, 7, 2, 5], [4, 6, 11, 10], [12, 0, 21, 13]], 4))
print(shift_grid_flatten([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 9))

[[9, 1, 2], [3, 4, 5], [6, 7, 8]]
[[12, 0, 21, 13], [3, 8, 1, 9], [19, 7, 2, 5], [4, 6, 11, 10]]
[[1, 2, 3], [4, 5, 6], [7, 8, 9]]


## 6. Cách 3 — Tính thẳng vị trí mới (lời giải chính)

Insight: phần tử ở vị trí `p` trong dãy dài sẽ đến vị trí `(p + k) % total`.

```text
p     = i * n + j               # 2D -> 1D
p_new = (p + k) % total         # tiến k bước, cuộn vòng
i2, j2 = divmod(p_new, n)       # 1D -> 2D
new[i2][j2] = grid[i][j]
```

Không cần dãy trung gian, mỗi phần tử được đặt đúng chỗ **một lần**.

In [9]:
# Ví dụ 1, k = 1: phần tử 9 ở (2, 2)
m, n, k = 3, 3, 1
i, j = 2, 2
p = i * n + j
p_new = (p + k) % (m * n)
print("p =", p, "-> p_new =", p_new, "-> (i2, j2) =", divmod(p_new, n))   # 8 -> 0 -> (0, 0)

p = 8 -> p_new = 0 -> (i2, j2) = (0, 0)


In [10]:
def shift_grid_index(grid, k):
    m, n = len(grid), len(grid[0])
    total = m * n
    new = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            p_new = (i * n + j + k) % total
            i2, j2 = divmod(p_new, n)
            new[i2][j2] = grid[i][j]
    return new

print(shift_grid_index([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1))
print(shift_grid_index([[3, 8, 1, 9], [19, 7, 2, 5], [4, 6, 11, 10], [12, 0, 21, 13]], 4))

[[9, 1, 2], [3, 4, 5], [6, 7, 8]]
[[12, 0, 21, 13], [3, 8, 1, 9], [19, 7, 2, 5], [4, 6, 11, 10]]


### Góc nhìn ngược: ô đích `(i, j)` lấy từ đâu?

Cũng có thể duyệt theo **ô đích**: ô ở vị trí `q` nhận phần tử từ vị trí `(q - k) % total`. Python xử lý `%` với số âm cho kết quả **không âm** (`-1 % 9 == 8`), rất tiện.

In [11]:
print(-1 % 9, -10 % 9)   # 8 8

def shift_grid_pull(grid, k):
    m, n = len(grid), len(grid[0])
    total = m * n
    return [[grid[(q := (i * n + j - k) % total) // n][q % n] for j in range(n)] for i in range(m)]

print(shift_grid_pull([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1))

8 8
[[9, 1, 2], [3, 4, 5], [6, 7, 8]]


`q := ...` là **walrus operator** (Python 3.8+): vừa gán `q` vừa dùng ngay trong biểu thức. Ở đây giúp tính `q` một lần rồi dùng cho cả `// n` và `% n`. Nếu thấy khó đọc, viết vòng `for` thường như Cách 3 là đủ.

## 7. Cách 4 — Đảo 3 lần (three reversals) và `deque.rotate`

Mẹo kinh điển (LeetCode 189 *Rotate Array*): xoay phải `k` bước một dãy = 
1. đảo **cả dãy**, 2. đảo `k` phần tử đầu, 3. đảo phần còn lại.

```text
[1 2 3 4 5], k=2
đảo cả:     [5 4 3 2 1]
đảo 2 đầu:  [4 5 | 3 2 1]
đảo phần sau: [4 5 | 1 2 3]   -> đúng bằng xoay phải 2
```
Ưu điểm: làm **tại chỗ** trên dãy, chỉ `O(1)` bộ nhớ phụ (nếu không tính dãy duỗi).

In [12]:
def reverse_range(a, lo, hi):
    # đảo a[lo..hi] (bao gồm hai đầu) bằng hai con trỏ
    while lo < hi:
        a[lo], a[hi] = a[hi], a[lo]
        lo += 1
        hi -= 1


def shift_grid_reverse(grid, k):
    m, n = len(grid), len(grid[0])
    flat = [x for row in grid for x in row]
    total = m * n
    k %= total
    reverse_range(flat, 0, total - 1)
    reverse_range(flat, 0, k - 1)
    reverse_range(flat, k, total - 1)
    return [flat[i:i + n] for i in range(0, total, n)]

print(shift_grid_reverse([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1))

[[9, 1, 2], [3, 4, 5], [6, 7, 8]]


`a[lo], a[hi] = a[hi], a[lo]` là **hoán đổi** hai phần tử trong một dòng (Python tính vế phải trước rồi mới gán).

`collections.deque` là hàng đợi hai đầu; phương thức `rotate(k)` xoay phải `k` bước (tự xử lý `k` lớn).

In [13]:
from collections import deque

d = deque([1, 2, 3, 4, 5])
d.rotate(2)
print(list(d))      # [4, 5, 1, 2, 3]

def shift_grid_deque(grid, k):
    n = len(grid[0])
    d = deque(x for row in grid for x in row)
    d.rotate(k)
    flat = list(d)
    return [flat[i:i + n] for i in range(0, len(flat), n)]

print(shift_grid_deque([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 10))   # như k=1

[4, 5, 1, 2, 3]
[[9, 1, 2], [3, 4, 5], [6, 7, 8]]


## 8. Trace từng bước (Cách 3)

Lưới `2 x 3` = `[[1,2,3],[4,5,6]]`, `k = 4`, `total = 6`.

| (i, j) | giá trị | `p = i*n+j` | `p_new = (p+4)%6` | `(i2, j2)` |
|---|---:|---:|---:|---|
| (0,0) | 1 | 0 | 4 | (1,1) |
| (0,1) | 2 | 1 | 5 | (1,2) |
| (0,2) | 3 | 2 | 0 | (0,0) |
| (1,0) | 4 | 3 | 1 | (0,1) |
| (1,1) | 5 | 4 | 2 | (0,2) |
| (1,2) | 6 | 5 | 3 | (1,0) |

Kết quả: `[[3,4,5],[6,1,2]]`. Kiểm tra bằng dãy dài: `[1..6]` xoay phải 4 → `[3,4,5,6,1,2]` ✔.

In [14]:
def trace_shift_grid(grid, k):
    m, n = len(grid), len(grid[0])
    total = m * n
    print(f"m={m}, n={n}, total={total}, k={k}, k%total={k % total}")
    new = [[None] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            p = i * n + j
            p_new = (p + k) % total
            i2, j2 = divmod(p_new, n)
            new[i2][j2] = grid[i][j]
            print(f"  ({i},{j}) val={grid[i][j]:>2} p={p} -> p_new={p_new} -> ({i2},{j2})  new={new}")
    return new

trace_shift_grid([[1, 2, 3], [4, 5, 6]], 4)

m=2, n=3, total=6, k=4, k%total=4
  (0,0) val= 1 p=0 -> p_new=4 -> (1,1)  new=[[None, None, None], [None, 1, None]]
  (0,1) val= 2 p=1 -> p_new=5 -> (1,2)  new=[[None, None, None], [None, 1, 2]]
  (0,2) val= 3 p=2 -> p_new=0 -> (0,0)  new=[[3, None, None], [None, 1, 2]]
  (1,0) val= 4 p=3 -> p_new=1 -> (0,1)  new=[[3, 4, None], [None, 1, 2]]
  (1,1) val= 5 p=4 -> p_new=2 -> (0,2)  new=[[3, 4, 5], [None, 1, 2]]
  (1,2) val= 6 p=5 -> p_new=3 -> (1,0)  new=[[3, 4, 5], [6, 1, 2]]


[[3, 4, 5], [6, 1, 2]]

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def shiftGrid(self, grid: list[list[int]], k: int) -> list[list[int]]:
        m, n = len(grid), len(grid[0])
        total = m * n
        new = [[0] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                i2, j2 = divmod((i * n + j + k) % total, n)
                new[i2][j2] = grid[i][j]
        return new

print(Solution().shiftGrid([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1))

[[9, 1, 2], [3, 4, 5], [6, 7, 8]]


Phiên bản ngắn (Cách 2) cũng được AC:

```python
class Solution:
    def shiftGrid(self, grid, k):
        n = len(grid[0])
        flat = [x for row in grid for x in row]
        k %= len(flat)
        flat = flat[-k:] + flat[:-k] if k else flat
        return [flat[i:i + n] for i in range(0, len(flat), n)]
```

## 10. Vì sao đúng?

- **Song ánh 2D ↔ 1D:** `(i, j) ↦ i·n + j` là ánh xạ 1-1 giữa các ô và `0..total-1` (đọc theo hàng). `divmod(p, n)` là ánh xạ ngược.
- **Một lần dịch = +1 trên dãy dài (mod total):** quy tắc 1 (`j → j+1`) là `p → p+1`; quy tắc 2 (`(i, n-1) → (i+1, 0)`) cũng là `p → p+1`; quy tắc 3 (`total-1 → 0`) là `p → (p+1) % total`.
- **k lần dịch = +k (mod total):** cộng dồn `k` lần `+1` theo modulo. Nên phần tử ở `p` kết thúc ở `(p + k) % total`.
- Vì phép `p ↦ (p+k) % total` cũng là song ánh, mỗi ô của `new` được ghi **đúng một lần** — không ô nào bị bỏ trống hay ghi đè.

## 11. Độ phức tạp và so sánh

Gọi `N = m·n`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng k lần | `O(k·N)` | `O(N)` | dễ hiểu, chậm khi k lớn |
| 2. Duỗi + slicing | `O(N)` | `O(N)` | ngắn gọn nhất |
| 3. Công thức chỉ số | `O(N)` | `O(N)` cho lưới kết quả | không cần dãy trung gian |
| 4a. Đảo 3 lần | `O(N)` | `O(1)` trên dãy duỗi | kỹ thuật tại chỗ |
| 4b. `deque.rotate` | `O(N)` | `O(N)` | dùng thư viện chuẩn |

Mọi cách đều phải trả ra lưới `m x n` nên `O(N)` bộ nhớ cho output là không tránh được.

## 12. Bộ test

Các nhóm: ví dụ đề; `k = 0`; `k` là bội của `N`; `k > N`; lưới 1 hàng; lưới 1 cột; lưới `1 x 1`; số âm. Cuối cùng đối chiếu tất cả các cách trên dữ liệu ngẫu nhiên.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}")

sol = Solution()
g1 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
check("đề 1", sol.shiftGrid(g1, 1), [[9, 1, 2], [3, 4, 5], [6, 7, 8]])
check("đề 2", sol.shiftGrid([[3, 8, 1, 9], [19, 7, 2, 5], [4, 6, 11, 10], [12, 0, 21, 13]], 4),
      [[12, 0, 21, 13], [3, 8, 1, 9], [19, 7, 2, 5], [4, 6, 11, 10]])
check("đề 3 (k = N)", sol.shiftGrid(g1, 9), g1)
check("k = 0", sol.shiftGrid(g1, 0), g1)
check("k > N", sol.shiftGrid(g1, 10), [[9, 1, 2], [3, 4, 5], [6, 7, 8]])
check("1 hàng", sol.shiftGrid([[1, 2, 3, 4]], 1), [[4, 1, 2, 3]])
check("1 cột", sol.shiftGrid([[1], [2], [3]], 1), [[3], [1], [2]])
check("1x1", sol.shiftGrid([[5]], 100), [[5]])
check("số âm", sol.shiftGrid([[-1, -2], [-3, -4]], 3), [[-2, -3], [-4, -1]])
print("grid gốc không bị đổi:", g1)

PASS đề 1
PASS đề 2
PASS đề 3 (k = N)
PASS k = 0
PASS k > N
PASS 1 hàng
PASS 1 cột
PASS 1x1
PASS số âm
grid gốc không bị đổi: [[1, 2, 3], [4, 5, 6], [7, 8, 9]]


In [17]:
import random
random.seed(1260)
methods = [shift_grid_simulate, shift_grid_flatten, shift_grid_index, shift_grid_pull,
           shift_grid_reverse, shift_grid_deque]
for _ in range(300):
    m, n = random.randint(1, 5), random.randint(1, 5)
    grid = [[random.randint(-9, 9) for _ in range(n)] for _ in range(m)]
    k = random.randint(0, 40)
    expected = shift_grid_simulate(grid, k)
    for f in methods:
        assert f(grid, k) == expected, (f.__name__, grid, k)
    assert Solution().shiftGrid(grid, k) == expected
print("300 bộ ngẫu nhiên: 6 cách + Solution đều khớp")

300 bộ ngẫu nhiên: 6 cách + Solution đều khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên `k %= total` khi dùng slicing

Nếu `k > total`, `flat[-k:]` lấy **cả dãy** và `flat[:-k]` rỗng → kết quả như không dịch gì.

In [18]:
def wrong_no_mod(grid, k):
    n = len(grid[0])
    flat = [x for row in grid for x in row]
    flat = flat[-k:] + flat[:-k]           # SAI khi k > len(flat)
    return [flat[i:i + n] for i in range(0, len(flat), n)]

g = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("sai :", wrong_no_mod(g, 10))
print("đúng:", shift_grid_flatten(g, 10))

sai : [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
đúng: [[9, 1, 2], [3, 4, 5], [6, 7, 8]]


### Lỗi 2: Tưởng `k = 0` làm hỏng `flat[-0:]`

`-0 == 0`, nên `flat[-0:]` là **cả dãy** và `flat[:-0]` là **rỗng**. Ghép lại vẫn là cả dãy — đúng *một cách tình cờ*. Viết rõ `if k:` để code không phụ thuộc may mắn và dễ đọc hơn.

In [19]:
flat = [1, 2, 3]
print(flat[-0:], flat[:-0], flat[-0:] + flat[:-0])   # [1, 2, 3] [] [1, 2, 3]

[1, 2, 3] [] [1, 2, 3]


### Lỗi 3: Dịch từng hàng riêng (quên phần tử "rơi" xuống hàng dưới)

Xoay từng hàng một cách độc lập là bài **khác**: phần tử cuối hàng phải xuống hàng kế, không phải quay về đầu chính hàng đó.

In [20]:
def wrong_rotate_each_row(grid, k):
    res = []
    for row in grid:
        kk = k % len(row)
        res.append(row[-kk:] + row[:-kk] if kk else row[:])
    return res

g = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("sai :", wrong_rotate_each_row(g, 1))   # [[3,1,2],[6,4,5],[9,7,8]]
print("đúng:", Solution().shiftGrid(g, 1))

sai : [[3, 1, 2], [6, 4, 5], [9, 7, 8]]
đúng: [[9, 1, 2], [3, 4, 5], [6, 7, 8]]


### Lỗi 4: Dùng `p // m` và `p % m` (chia cho số hàng)

Vị trí 1 chiều được tạo bằng `i * n + j` (**n = số cột**) thì phải chia lại cho **n**. Với lưới vuông lỗi bị che giấu; lưới `2 x 3` lộ ngay (thậm chí `IndexError`).

In [21]:
def wrong_divide_by_m(grid, k):
    m, n = len(grid), len(grid[0])
    total = m * n
    new = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            i2, j2 = divmod((i * n + j + k) % total, m)    # SAI: phải là n
            new[i2][j2] = grid[i][j]
    return new

try:
    print(wrong_divide_by_m([[1, 2, 3], [4, 5, 6]], 1))
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### Lỗi 5: Sửa trực tiếp `grid` trong lúc đọc

Ghi `grid[i2][j2] = grid[i][j]` ngay trên lưới gốc sẽ ghi đè giá trị chưa kịp đọc. Hãy ghi vào lưới **mới** (hoặc dùng kỹ thuật đảo 3 lần có kiểm soát).

In [22]:
def wrong_in_place(grid, k):
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]          # bản sao để không phá dữ liệu test
    for i in range(m):
        for j in range(n):
            i2, j2 = divmod((i * n + j + k) % (m * n), n)
            g[i2][j2] = g[i][j]           # SAI: đọc từ lưới đang bị ghi đè
    return g

print("sai :", wrong_in_place([[1, 2, 3], [4, 5, 6]], 1))
print("đúng:", Solution().shiftGrid([[1, 2, 3], [4, 5, 6]], 1))

sai : [[1, 1, 1], [1, 1, 1]]
đúng: [[6, 1, 2], [3, 4, 5]]


`row[:]` tạo **bản sao nông** của một hàng — cách nhanh để copy ma trận số: `[row[:] for row in grid]`.

## 14. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:**
- "Phần tử cuối quay về đầu", "dịch vòng", "xoay k lần" → **rotate** + `k % length`.
- Ma trận nhưng thứ tự di chuyển theo kiểu đọc sách (row-major) → **duỗi thành 1 chiều** với `p = i*n + j`.
- `k` có thể rất lớn so với kích thước → luôn rút gọn `k %= N` trước.

**Bài liên quan:**
- LeetCode 189 *Rotate Array* (xoay dãy, đảo 3 lần).
- LeetCode 61 *Rotate List* (xoay danh sách liên kết).
- LeetCode 796 *Rotate String* (`goal in s + s`).
- LeetCode 566 *Reshape the Matrix* (đổi `m x n` → `r x c` qua chỉ số 1 chiều).
- LeetCode 48 *Rotate Image* (xoay hình học 90° — khác hẳn bài này!).

In [23]:
# Mini: Reshape the Matrix (566) dùng cùng ý tưởng chỉ số 1 chiều
def reshape(mat, r, c):
    m, n = len(mat), len(mat[0])
    if m * n != r * c:
        return mat
    res = [[0] * c for _ in range(r)]
    for p in range(m * n):
        res[p // c][p % c] = mat[p // n][p % n]
    return res

print(reshape([[1, 2], [3, 4]], 1, 4))   # [[1, 2, 3, 4]]
print(reshape([[1, 2], [3, 4]], 4, 1))   # [[1], [2], [3], [4]]

[[1, 2, 3, 4]]
[[1], [2], [3], [4]]


## 15. Bài tập tự luyện

> Gợi ý ở mục 16, đáp án ở mục 17. Cell bài tập chỉ định nghĩa hàm, **không** gọi.

### Bài 1 — Đoán output
`[1, 2, 3, 4, 5, 6][-4:] + [1, 2, 3, 4, 5, 6][:-4]` cho ra gì? Gán vào `ex1_answer`.

In [24]:
ex1_answer = None   # TODO

# assert ex1_answer == [1, 2, 3, 4, 5, 6][-4:] + [1, 2, 3, 4, 5, 6][:-4]

### Bài 2 — Dịch trái
Viết `shift_grid_left(grid, k)`: mỗi phần tử lùi 1 vị trí (phần tử đầu quay về cuối), làm `k` lần.

In [25]:
def shift_grid_left(grid: list[list[int]], k: int) -> list[list[int]]:
    # TODO
    pass

# assert shift_grid_left([[1, 2, 3], [4, 5, 6]], 1) == [[2, 3, 4], [5, 6, 1]]
# assert shift_grid_left([[1, 2], [3, 4]], 5) == [[2, 3], [4, 1]]

### Bài 3 — Phần tử nào nằm ở ô (i, j)?
Không dựng cả lưới mới, viết `value_after_shift(grid, k, i, j)` trả về giá trị ở ô `(i, j)` sau khi dịch `k` lần, trong `O(1)`.

In [26]:
def value_after_shift(grid: list[list[int]], k: int, i: int, j: int) -> int:
    # TODO: ô đích q = i*n + j nhận phần tử từ vị trí (q - k) % total
    pass

# assert value_after_shift([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1, 0, 0) == 9
# assert value_after_shift([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 4, 2, 2) == 5

### Bài 4 — Xoay tại chỗ
Viết `rotate_in_place(nums, k)` xoay phải list `nums` **tại chỗ** (không tạo list mới), bằng kỹ thuật đảo 3 lần. Hàm không trả gì.

In [27]:
def rotate_in_place(nums: list[int], k: int) -> None:
    # TODO
    pass

# a = [1, 2, 3, 4, 5, 6, 7]; rotate_in_place(a, 3); assert a == [5, 6, 7, 1, 2, 3, 4]
# a = [1, 2]; rotate_in_place(a, 5); assert a == [2, 1]

### Bài 5 — Mini challenge: dịch theo cột
Dịch "đọc theo cột" (column-major): đọc từ trên xuống dưới từng cột, trái sang phải; dịch phải `k` bước theo thứ tự đó. Viết `shift_grid_column_major(grid, k)`.

In [28]:
def shift_grid_column_major(grid: list[list[int]], k: int) -> list[list[int]]:
    # TODO: vị trí 1 chiều p = j*m + i
    pass

# assert shift_grid_column_major([[1, 3], [2, 4]], 1) == [[4, 2], [1, 3]]
# assert shift_grid_column_major([[1, 4], [2, 5], [3, 6]], 2) == [[5, 2], [6, 3], [1, 4]]

## 16. Gợi ý 3 mức

**Bài 1:** (1) `a[-4:]` = 4 phần tử cuối. (2) `a[:-4]` = 2 phần tử đầu. (3) Ghép: `[3,4,5,6] + [1,2]`.

**Bài 2:** (1) Dịch trái k = dịch phải `total - k`. (2) Hoặc slicing `flat[k:] + flat[:k]`. (3) Nhớ `k %= total` trước.

**Bài 3:** (1) Nghĩ "ô đích lấy từ đâu". (2) `q = i*n + j`, `src = (q - k) % total`. (3) `grid[src // n][src % n]`.

**Bài 4:** (1) Mục 7. (2) Viết hàm phụ đảo đoạn `[lo, hi]`. (3) `k %= len`; đảo cả, đảo `[0, k-1]`, đảo `[k, len-1]`.

**Bài 5:** (1) Duỗi theo cột: `[grid[i][j] for j in range(n) for i in range(m)]`. (2) Xoay như cũ. (3) Ghi lại: phần tử thứ `p` vào `grid2[p % m][p // m]`.

## 17. Đáp án tham khảo

In [29]:
ans_ex1 = [3, 4, 5, 6, 1, 2]


def ans_shift_grid_left(grid, k):
    n = len(grid[0])
    flat = [x for row in grid for x in row]
    k %= len(flat)
    flat = flat[k:] + flat[:k]
    return [flat[i:i + n] for i in range(0, len(flat), n)]


def ans_value_after_shift(grid, k, i, j):
    m, n = len(grid), len(grid[0])
    total = m * n
    src = (i * n + j - k) % total
    return grid[src // n][src % n]


def ans_rotate_in_place(nums, k):
    L = len(nums)
    k %= L
    def rev(lo, hi):
        while lo < hi:
            nums[lo], nums[hi] = nums[hi], nums[lo]
            lo += 1
            hi -= 1
    rev(0, L - 1)
    rev(0, k - 1)
    rev(k, L - 1)


def ans_shift_grid_column_major(grid, k):
    m, n = len(grid), len(grid[0])
    flat = [grid[i][j] for j in range(n) for i in range(m)]
    k %= len(flat)
    if k:
        flat = flat[-k:] + flat[:-k]
    res = [[0] * n for _ in range(m)]
    for p, v in enumerate(flat):
        res[p % m][p // m] = v
    return res


assert ans_ex1 == [1, 2, 3, 4, 5, 6][-4:] + [1, 2, 3, 4, 5, 6][:-4]
assert ans_shift_grid_left([[1, 2, 3], [4, 5, 6]], 1) == [[2, 3, 4], [5, 6, 1]]
assert ans_shift_grid_left([[1, 2], [3, 4]], 5) == [[2, 3], [4, 1]]
assert ans_value_after_shift([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1, 0, 0) == 9
assert ans_value_after_shift([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 4, 2, 2) == 5
a = [1, 2, 3, 4, 5, 6, 7]; ans_rotate_in_place(a, 3); assert a == [5, 6, 7, 1, 2, 3, 4]
a = [1, 2]; ans_rotate_in_place(a, 5); assert a == [2, 1]
assert ans_shift_grid_column_major([[1, 3], [2, 4]], 1) == [[4, 2], [1, 3]]
assert ans_shift_grid_column_major([[1, 4], [2, 5], [3, 6]], 2) == [[5, 2], [6, 3], [1, 4]]

# đối chiếu thêm: value_after_shift khớp toàn bộ lưới của Solution
import random
random.seed(5)
for _ in range(100):
    m, n = random.randint(1, 4), random.randint(1, 4)
    g = [[random.randint(0, 50) for _ in range(n)] for _ in range(m)]
    k = random.randint(0, 30)
    full = Solution().shiftGrid(g, k)
    assert all(ans_value_after_shift(g, k, i, j) == full[i][j] for i in range(m) for j in range(n))
    assert ans_shift_grid_left(Solution().shiftGrid(g, k), k) == g     # dịch phải rồi trái = gốc
print("Đáp án bài tập: tất cả assert PASS")

Đáp án bài tập: tất cả assert PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def shiftGrid(self, grid, k):
        m, n = len(grid), len(grid[0])
        total = m * n
        new = [[0] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                i2, j2 = divmod((i * n + j + k) % total, n)
                new[i2][j2] = grid[i][j]
        return new
```

**Câu thần chú:** *"Duỗi lưới thành dãy (p = i·n + j), tiến k bước modulo N, gấp lại bằng divmod(p, n)."*

**Checklist:**
- [ ] Đã rút gọn `k %= m*n` (hoặc dùng `% total` trong công thức).
- [ ] Chia cho **n (số cột)**, không phải `m`.
- [ ] Ghi vào lưới **mới**, không ghi đè lưới đang đọc.
- [ ] Phần tử cuối hàng xuống **hàng kế**, không quay về đầu cùng hàng.
- [ ] Test `k = 0`, `k = N`, `k > N`, lưới 1 hàng / 1 cột.